# Suite TSPRT — Printing TypeScript

The standards print TypeScript trees as source text in one fixed layout (`TypeScript._Printer`). Trees built by hand
carry no parentheses, so the printer adds those the grammar needs. Each case prints hand-built trees and checks the
text, which parses back to the same tree but for the parentheses added.

In [ ]:
from mbse.Programs.Framework import Syntax as F
from mbse.Programs.Framework.Errors import PrintError
from mbse.Programs.TypeScript import ECMAScript2025, Syntax as S, TypeScript59
from mbse.Programs.TypeScript._Printer import Printer
from support import raises

TS = TypeScript59.STANDARD
TSX = TypeScript59.JSX


def n(name):
    return S.Identifier(name=name)


def lit(raw):
    return S.Literal(raw=raw)


def ref(name):
    return S.TSTypeReference(typeName=n(name))


def binary(left, operator, right):
    kind = S.LogicalExpression if operator in ("&&", "||", "??") else S.BinaryExpression
    return kind(left=left, operator=operator, right=right)


def strip(node):
    """`node` without the parentheses the source wrote."""
    if isinstance(node, S.ParenthesizedExpression):
        return strip(node.expression)
    if isinstance(node, S.TSParenthesizedType):
        return strip(node.typeAnnotation)
    if isinstance(node, list):
        return [strip(x) for x in node]
    if isinstance(node, F.Node):
        for field in node.FIELDS:
            setattr(node, field.name, strip(getattr(node, field.name)))
    return node


def unshare(node):
    """`node` with each place holding a node of its own: hand-built trees here reuse a few names."""
    if isinstance(node, list):
        return [unshare(x) for x in node]
    if isinstance(node, F.Node):
        return type(node)(**{field.name: unshare(getattr(node, field.name)) for field in node.FIELDS})
    return node


def prints(node, expected, standard=TS):
    """`node` prints as `expected`, which parses back to `node` but for parentheses."""
    node = unshare(node)
    text = standard.print(node)
    assert text == expected, text
    if isinstance(node, S.Expression) and not isinstance(node, S.Identifier):
        back = strip(standard.parse(f"({expected});").body[0].expression)
        assert F.same(back, node), (back, node)

## TSPRT-01 · Precedence: operands are parenthesized where they bind less than their place needs

In [ ]:
a, b, c = n("a"), n("b"), n("c")
prints(binary(binary(a, "+", b), "*", c), "(a + b) * c")
prints(binary(a, "-", binary(b, "-", c)), "a - (b - c)")
prints(binary(binary(a, "-", b), "-", c), "a - b - c")
prints(binary(a, "**", binary(b, "**", c)), "a ** b ** c")
prints(binary(binary(a, "**", b), "**", c), "(a ** b) ** c")
prints(binary(S.UnaryExpression(operator="-", argument=a), "**", b), "(-a) ** b")
prints(binary(S.AwaitExpression(argument=a), "**", b), "(await a) ** b")
prints(binary(binary(a, "??", b), "||", c), "(a ?? b) || c")
prints(binary(a, "??", binary(b, "&&", c)), "a ?? (b && c)")
prints(binary(binary(a, "||", b), "??", c), "(a || b) ?? c")
prints(binary(binary(a, "&&", b), "||", c), "a && b || c")
prints(binary(S.PrivateIdentifier(name="x"), "in", a), "#x in a")
prints(S.ConditionalExpression(test=S.ConditionalExpression(test=a, consequent=b, alternate=c), consequent=b,
                               alternate=S.SequenceExpression(expressions=[a, b])), "(a ? b : c) ? b : (a, b)")
prints(S.AssignmentExpression(left=a, operator="=", right=S.AssignmentExpression(left=b, operator="+=", right=c)),
       "a = b += c")
prints(S.AssignmentExpression(left=S.ObjectPattern(properties=[
    S.Property(propertyKind="init", key=a, value=n("a"), shorthand=True)]),
                              operator="=", right=b), "{ a } = b")
prints(S.UnaryExpression(operator="-", argument=S.UnaryExpression(operator="-", argument=a)), "- -a")
prints(S.UnaryExpression(operator="+", argument=S.UpdateExpression(operator="++", prefix=True, argument=a)), "+ ++a")
prints(S.UnaryExpression(operator="-", argument=S.UnaryExpression(operator="+", argument=a)), "-+a")
prints(S.UnaryExpression(operator="typeof", argument=binary(a, "+", b)), "typeof (a + b)")
prints(S.UnaryExpression(operator="!", argument=S.UnaryExpression(operator="!", argument=a)), "!!a")
prints(S.UpdateExpression(operator="++", prefix=False, argument=S.MemberExpression(object=a, property=b)), "a.b++")
prints(S.UpdateExpression(operator="--", prefix=True, argument=a), "--a")
prints(S.MemberExpression(object=binary(a, "+", b), property=c), "(a + b).c")
prints(S.MemberExpression(object=lit("1"), property=n("toString")), "(1).toString")
prints(S.MemberExpression(object=lit("1.5"), property=n("toFixed")), "1.5.toFixed")
prints(S.MemberExpression(object=S.NewExpression(callee=a), property=b), "new a().b")
prints(S.ChainExpression(expression=S.MemberExpression(object=a, property=binary(b, "+", c), computed=True,
                                                       optional=True)), "a?.[b + c]")
prints(S.CallExpression(callee=S.ArrowFunctionExpression(body=a), arguments=[S.SequenceExpression(
    expressions=[a, b])]), "(() => a)((a, b))")
prints(S.NewExpression(callee=S.CallExpression(callee=a)), "new (a())()")
prints(S.NewExpression(callee=S.MemberExpression(object=S.CallExpression(callee=a), property=b)), "new (a().b)()")
prints(S.NewExpression(callee=S.MemberExpression(object=S.TSNonNullExpression(expression=a), property=b)),
       "new (a!.b)()")
prints(S.NewExpression(callee=S.TaggedTemplateExpression(tag=a, quasi=S.TemplateLiteral(
    quasis=[S.TemplateElement(raw="x", tail=True)]))), "new a`x`()")
prints(S.NewExpression(callee=S.NewExpression(callee=a)), "new (new a())()")
prints(S.TSAsExpression(expression=binary(a, "??", b), typeAnnotation=ref("T")), "(a ?? b) as T")
prints(binary(S.TSAsExpression(expression=a, typeAnnotation=ref("T")), "+", b), "(a as T) + b")
prints(S.TSNonNullExpression(expression=S.AwaitExpression(argument=a)), "(await a)!")
prints(S.TSTypeAssertion(typeAnnotation=ref("T"), expression=binary(a, "+", b)), "<T>(a + b)")
prints(S.AwaitExpression(argument=S.TSTypeAssertion(typeAnnotation=ref("T"), expression=a)), "await <T>a")
prints(S.YieldExpression(argument=S.SequenceExpression(expressions=[a, b])), "yield (a, b)")
prints(S.SpreadElement(argument=S.SequenceExpression(expressions=[a, b])), "...(a, b)")
prints(S.ArrayExpression(elements=[S.Elision(), a, S.Elision(), S.Elision()]), "[, a,,,]")
prints(S.ArrayExpression(elements=[a, S.Elision(), lit("', ,'")]), "[a,, ', ,']")
print("ok")

## TSPRT-02 · Where an expression starts a statement, an arrow's body, a `for` or a default export

In [ ]:
def statement(expression):
    return S.ExpressionStatement(expression=expression)


prints(statement(S.ObjectExpression()), "({});")
prints(statement(S.CallExpression(callee=S.FunctionExpression(body=S.BlockStatement()))), "(function() {}());")
prints(statement(S.MemberExpression(object=S.ClassExpression(body=S.ClassBody()), property=a)), "(class {}.a);")
prints(statement(S.AssignmentExpression(left=S.ObjectPattern(), operator="=", right=a)), "({} = a);")
prints(statement(S.AssignmentExpression(left=S.MemberExpression(object=n("let"), property=a, computed=True),
                                        operator="=", right=b)), "(let[a] = b);")
prints(statement(S.UpdateExpression(operator="++", argument=S.MemberExpression(object=S.ObjectExpression(),
                                                                                 property=a))), "({}.a++);")
prints(statement(S.TSAsExpression(expression=S.ObjectExpression(), typeAnnotation=ref("T"))), "({} as T);")
prints(statement(S.ConditionalExpression(test=S.TaggedTemplateExpression(
    tag=S.FunctionExpression(body=S.BlockStatement()), quasi=S.TemplateLiteral(
        quasis=[S.TemplateElement(raw="", tail=True)])), consequent=a, alternate=b)), "(function() {}`` ? a : b);")
prints(S.ArrowFunctionExpression(body=S.ObjectExpression()), "() => ({})")
prints(S.ArrowFunctionExpression(body=S.MemberExpression(object=S.ObjectExpression(), property=a)), "() => ({}.a)")
prints(S.ArrowFunctionExpression(body=S.SequenceExpression(expressions=[a, b])), "() => (a, b)")
prints(S.ArrowFunctionExpression(params=[a], body=S.ArrowFunctionExpression(body=a)), "(a) => () => a")
in_init = S.ForStatement(init=S.VariableDeclaration(declarationKind="let", declarations=[S.VariableDeclarator(
    id=a, init=binary(lit("'x'"), "in", b))]), body=S.EmptyStatement())
prints(in_init, "for (let a = ('x' in b);;)\n    ;")
for init in (S.ConditionalExpression(test=binary(a, "in", b), consequent=a, alternate=b),
             S.SequenceExpression(expressions=[a, binary(a, "in", b)]),
             S.TSAsExpression(expression=binary(a, "in", b), typeAnnotation=ref("T")),
             S.ArrowFunctionExpression(body=binary(a, "in", b)), binary(a, "&&", binary(a, "in", b)),
             S.AssignmentExpression(left=a, operator="=", right=binary(a, "in", b)),
             S.ConditionalExpression(test=a, consequent=b, alternate=binary(a, "in", b))):
    text = TS.print(unshare(S.ForStatement(init=init, body=S.EmptyStatement())))
    assert text.startswith("for ((") and ";;)" in text, text
assert TS.print(unshare(S.ForStatement(init=binary(a, "+", b), body=S.EmptyStatement()))) == "for (a + b;;)\n    ;"
prints(S.ForOfStatement(left=n("async"), right=a, body=S.EmptyStatement()), "for ((async) of a)\n    ;")
prints(S.ForOfStatement(isAwait=True, left=n("async"), right=a, body=S.EmptyStatement()),
       "for await (async of a)\n    ;")
prints(S.ForOfStatement(left=a, right=S.SequenceExpression(expressions=[a, b]), body=S.EmptyStatement()),
       "for (a of (a, b))\n    ;")
prints(S.ForInStatement(left=S.MemberExpression(object=a, property=b), right=c, body=S.BlockStatement()),
       "for (a.b in c) {}")
prints(S.ExportDefaultDeclaration(declaration=S.FunctionExpression(body=S.BlockStatement())),
       "export default (function() {});")
prints(S.ExportDefaultDeclaration(declaration=S.CallExpression(callee=S.ClassExpression(body=S.ClassBody()))),
       "export default (class {}());")
prints(S.ExportDefaultDeclaration(declaration=S.SequenceExpression(expressions=[a, b])), "export default (a, b);")
prints(S.ExportDefaultDeclaration(declaration=S.ArrowFunctionExpression(body=a)), "export default () => a;")
print("ok")

## TSPRT-03 · Types: parentheses where a function, conditional, union or operator type is an operand

In [ ]:
T, U = ref("T"), ref("U")
function = S.TSFunctionType(returnType=S.TSTypeAnnotation(typeAnnotation=T))
union = S.TSUnionType(types=[T, U])
conditional = S.TSConditionalType(checkType=T, extendsType=U, trueType=T, falseType=U)
prints(S.TSArrayType(elementType=union), "(T | U)[]")
prints(S.TSUnionType(types=[function, T]), "(() => T) | T")
prints(S.TSIntersectionType(types=[union, T]), "(T | U) & T")
prints(S.TSUnionType(types=[S.TSIntersectionType(types=[T, U]), T]), "T & U | T")
prints(S.TSTypeOperator(operator="keyof", typeAnnotation=union), "keyof (T | U)")
prints(S.TSArrayType(elementType=S.TSTypeOperator(operator="keyof", typeAnnotation=T)), "(keyof T)[]")
prints(S.TSIndexedAccessType(objectType=union, indexType=union), "(T | U)[T | U]")
prints(S.TSOptionalType(typeAnnotation=union), "(T | U)?")
prints(S.TSConditionalType(checkType=conditional, extendsType=conditional, trueType=conditional,
                           falseType=conditional),
       "(T extends U ? T : U) extends (T extends U ? T : U) ? T extends U ? T : U : T extends U ? T : U")
prints(S.TSConditionalType(checkType=function, extendsType=function, trueType=T, falseType=U),
       "(() => T) extends () => T ? T : U")
returns_conditional = S.TSFunctionType(returnType=S.TSTypeAnnotation(typeAnnotation=conditional))
prints(S.TSConditionalType(checkType=T, extendsType=returns_conditional, trueType=T, falseType=U),
       "T extends (() => T extends U ? T : U) ? T : U")
prints(S.TSConditionalType(checkType=T, extendsType=S.TSConstructorType(), trueType=T, falseType=U),
       "T extends new () ? T : U")
prints(S.TSTypeOperator(operator="readonly", typeAnnotation=S.TSArrayType(elementType=T)), "readonly T[]")
prints(S.TSTypeOperator(operator="unique", typeAnnotation=S.TSSymbolKeyword()), "unique symbol")
prints(S.TSInferType(typeParameter=S.TSTypeParameter(name=n("X"), constraint=union)), "infer X extends T | U")
prints(S.TSArrayType(elementType=S.TSInferType(typeParameter=S.TSTypeParameter(name=n("X")))), "(infer X)[]")
print("ok")

## TSPRT-04 · Every kind prints on its own, as the text it stands for

In [ ]:
texts = {
    S.Comment(text=" a"): "// a", S.Comment(block=True, text=" a "): "/* a */", S.ThisExpression(): "this",
    S.Super(): "super", S.Elision(): "", S.TemplateElement(raw="a"): "a",
    S.ParenthesizedExpression(expression=a): "(a)", S.MetaProperty(meta=n("new"), property=n("target")): "new.target",
    S.ChainExpression(expression=S.CallExpression(callee=a, optional=True)): "a?.()",
    S.ImportExpression(source=lit("'m'"), options=a): "import('m', a)",
    S.ImportExpression(source=lit("'m'"), phase="defer"): "import.defer('m')",
    S.TSInstantiationExpression(expression=a, typeArguments=S.TSTypeParameterInstantiation(params=[T])): "a<T>",
    S.TSSatisfiesExpression(expression=a, typeAnnotation=T): "a satisfies T",
    S.Decorator(expression=S.CallExpression(callee=a)): "@a()",
    S.ImportAttribute(key=n("type"), value=lit("'json'")): "type: 'json'",
    S.ImportSpecifier(importKind="value", imported=a, local=n("a")): "a",
    S.ImportSpecifier(importKind="type", imported=a, local=b):
        "type a as b",
    S.ImportDefaultSpecifier(local=a): "a", S.ImportNamespaceSpecifier(local=a): "* as a",
    S.ExportSpecifier(exportKind="value", local=a, exported=n("a")): "a",
    S.ExportSpecifier(exportKind="type", local=a, exported=b):
        "type a as b",
    S.TSExternalModuleReference(expression=lit("'m'")): "require('m')",
    S.TSTypeAnnotation(typeAnnotation=T): "T",
    S.TSTypeParameter(const=True, isIn=True, isOut=True, name=n("X"), constraint=T, default=U):
        "const in out X extends T = U",
    S.TSTypeParameterDeclaration(params=[S.TSTypeParameter(name=n("X"))]): "<X>",
    S.TSInterfaceHeritage(expression=S.MemberExpression(object=a, property=b)): "a.b",
    S.TSClassImplements(expression=a, typeArguments=S.TSTypeParameterInstantiation(params=[T])): "a<T>",
    S.TSEnumMember(id=lit("'a'"), initializer=lit("1")): "'a' = 1",
    S.TSEnumMember(id=a, computed=True): "[a]",
    S.TSPropertySignature(readonly=True, key=a, optional=True, typeAnnotation=S.TSTypeAnnotation(typeAnnotation=T)):
        "readonly a?: T;",
    S.TSPropertySignature(key=a, computed=True): "[a];",
    S.TSMethodSignature(methodKind="get", key=a): "get a();",
    S.TSCallSignatureDeclaration(params=[a]): "(a);",
    S.TSConstructSignatureDeclaration(typeParameters=S.TSTypeParameterDeclaration(
        params=[S.TSTypeParameter(name=n("X"))])): "new <X>();",
    S.TSIndexSignature(accessibility="private", static=True, readonly=True,
                       parameters=[S.Identifier(name="k", typeAnnotation=S.TSTypeAnnotation(
                           typeAnnotation=S.TSStringKeyword()))]): "private static readonly [k: string];",
    S.TSIntrinsicKeyword(): "intrinsic", S.TSThisType(): "this", S.TSQualifiedName(left=a, right=b): "a.b",
    S.TSParenthesizedType(typeAnnotation=union): "(T | U)", S.TSLiteralType(literal=lit("'a'")): "'a'",
    S.TSTemplateLiteralType(quasis=[S.TemplateElement(raw="a"), S.TemplateElement(raw="b", tail=True)], types=[T]):
        "`a${T}b`",
    S.TSTupleType(elementTypes=[S.TSNamedTupleMember(label=a, optional=True, elementType=T),
                                S.TSRestType(typeAnnotation=S.TSArrayType(elementType=U))]): "[a?: T, ...U[]]",
    S.TSTypeLiteral(): "{}",
    S.TSMappedType(key=n("K"), constraint=T): "{ [K in T] }",
    S.TSTypeQuery(exprName=S.TSImportType(source=lit("'m'")), typeArguments=S.TSTypeParameterInstantiation(
        params=[T])): "typeof import('m')<T>",
    S.TSImportType(source=lit("'m'"), options=S.ObjectExpression(), qualifier=a): "import('m', {}).a",
    S.TSTypePredicate(asserts=True, parameterName=S.TSThisType()): "asserts this",
    S.TSTypePredicate(parameterName=a, typeAnnotation=S.TSTypeAnnotation(typeAnnotation=T)): "a is T",
    S.TSConstructorType(abstract=True, params=[a], returnType=S.TSTypeAnnotation(typeAnnotation=T)):
        "abstract new (a) => T",
    S.TSTypeOperator(operator="keyof"): "keyof",
    S.ObjectPattern(optional=True, typeAnnotation=S.TSTypeAnnotation(typeAnnotation=T)): "{}?: T",
    S.ArrayPattern(elements=[a, S.Elision()], decorators=[S.Decorator(expression=a)]): "@a [a,,]",
    S.RestElement(argument=a, typeAnnotation=S.TSTypeAnnotation(typeAnnotation=T)): "...a: T",
    S.AssignmentPattern(left=a, right=S.SequenceExpression(expressions=[a, b])): "a = (a, b)",
    S.TSParameterProperty(decorators=[S.Decorator(expression=a)], accessibility="private", readonly=True,
                          override=True, parameter=b): "@a private override readonly b",
    S.Property(propertyKind="init", key=a, value=S.AssignmentPattern(left=n("a"), right=lit("1")), shorthand=True):
        "a = 1",
    S.Property(propertyKind="init", key=a, value=S.TSEmptyBodyFunctionExpression(), method=True): "a()",
    S.Property(propertyKind="init", key=a, computed=True, optional=True, value=S.FunctionExpression(
        isAsync=True, generator=True, body=S.BlockStatement()), method=True): "async *[a]?() {}",
    S.Property(propertyKind="get", key=a, value=S.FunctionExpression(body=S.BlockStatement())): "get a() {}",
    S.YieldExpression(delegate=True, argument=a): "yield* a", S.YieldExpression(): "yield",
    S.ClassExpression(decorators=[S.Decorator(expression=a)], id=n("C"), body=S.ClassBody()): "@a class C {}",
    S.JSXIdentifier(name="a-b"): "a-b", S.JSXEmptyExpression(): "",
    S.JSXSpreadAttribute(argument=S.SequenceExpression(expressions=[a, b])): "{...(a, b)}",
    S.JSXSpreadChild(expression=a): "{...a}", S.JSXText(raw=" a "): " a ",
    S.JSXAttribute(name=S.JSXNamespacedName(namespace=S.JSXIdentifier(name="a"), name=S.JSXIdentifier(name="b"))):
        "a:b",
    S.JSXFragment(openingFragment=S.JSXOpeningFragment(), closingFragment=S.JSXClosingFragment()): "<></>",
    S.SwitchCase(consequent=[S.Comment(text=" c", trailing=True), S.BreakStatement()]): "default: // c\n    break;",
    S.MethodDefinition(methodKind="method", key=a, value=S.TSEmptyBodyFunctionExpression()): "a();",
    S.TSAbstractMethodDefinition(accessibility="protected", methodKind="get", key=a,
                                 value=S.TSEmptyBodyFunctionExpression()): "protected abstract get a();",
    S.StaticBlock(): "static {}",
    S.TSDeclareFunction(id=a): "function a();",
    S.ClassDeclaration(declare=True, abstract=True, id=n("C"), body=S.ClassBody()): "declare abstract class C {}",
    S.TSInterfaceDeclaration(declare=True, id=n("I"), body=S.TSInterfaceBody()): "declare interface I {}",
    S.TSEnumDeclaration(id=n("E"), body=S.TSEnumBody()): "enum E {}",
    S.TSModuleDeclaration(moduleKind="global", id=n("global"), body=S.TSModuleBlock()): "global {}",
    S.TSTypeAliasDeclaration(declare=True, id=n("X"), typeAnnotation=T): "declare type X = T;",
    S.VariableDeclaration(declare=True, declarationKind="var", declarations=[S.VariableDeclarator(id=a)]):
        "declare var a;",
    S.ImportDeclaration(importKind="type", source=lit("'m'"), specifiers=[
        S.ImportDefaultSpecifier(local=a), S.ImportNamespaceSpecifier(local=b)]): "import type a, * as b from 'm';",
    S.ExportAllDeclaration(exportKind="type", source=lit("'m'"), attributes=[
        S.ImportAttribute(key=n("type"), value=lit("'json'"))]): "export type * from 'm' with { type: 'json' };",
    S.TSImportEqualsDeclaration(importKind="type", id=a, moduleReference=n("b")): "import type a = b;",
    S.ExportNamedDeclaration(exportKind="value", declaration=S.ClassDeclaration(
        decorators=[S.Decorator(expression=a)], id=n("C"),
                                                            body=S.ClassBody())): "@a\nexport class C {}",
    S.ReturnStatement(argument=S.SequenceExpression(expressions=[a, b])): "return a, b;",
    S.ThrowStatement(argument=a): "throw a;", S.ContinueStatement(): "continue;", S.BreakStatement(label=a):
        "break a;",
    S.LabeledStatement(label=a, body=S.LabeledStatement(label=b, body=S.EmptyStatement())): "a: b: ;",
    S.WithStatement(object=a, body=S.ExpressionStatement(expression=b)): "with (a)\n    b;",
}
for node, text in texts.items():
    standard = TSX if type(node).KIND.startswith("JSX") else TS
    assert standard.print(unshare(node)) == text, (node, standard.print(unshare(node)))
assert Printer().print(S.TSTypeAnnotation(typeAnnotation=T)) == "T"
print(len(texts), "kinds")

## TSPRT-05 · Layout: blocks, `else if`, members, comments, and functions within expressions

In [ ]:
def block(*body):
    return S.BlockStatement(body=list(body))


def say(text):
    return S.ExpressionStatement(expression=n(text))


nested = S.IfStatement(test=a, consequent=block(say("b")), alternate=S.IfStatement(
    test=b, consequent=say("c"), alternate=block(S.IfStatement(test=c, consequent=block(), alternate=say("d")))))
assert TS.print(unshare(S.Program(body=[nested]))) == """if (a) {
    b;
} else if (b)
    c;
else {
    if (c) {} else
        d;
}
"""
assert TS.print(unshare(S.IfStatement(test=a, consequent=say("b"), alternate=block()))) == "if (a)\n    b;\nelse {}"
assert TS.print(unshare(S.IfStatement(test=a, consequent=block()))) == "if (a) {}"
assert TS.print(unshare(S.TryStatement(block=block(), finalizer=block(say("b"))))) == "try {} finally {\n    b;\n}"
assert TS.print(unshare(S.ClassDeclaration(id=n("C"), superClass=b, body=S.ClassBody()))) == "class C extends b {}"
assert TS.print(unshare(S.DoWhileStatement(body=say("a"), test=b))) == "do\n    a;\nwhile (b);"
inner = S.VariableDeclaration(declarationKind="const", declarations=[S.VariableDeclarator(
    id=a, init=S.ArrowFunctionExpression(body=block(S.ReturnStatement(argument=S.ClassExpression(body=S.ClassBody(
        body=[S.MethodDefinition(methodKind="method", key=b, value=S.FunctionExpression(body=block(say("c"))))]))))))])
function = S.FunctionDeclaration(id=n("f"), body=block(inner, S.Comment(text=" trailing", trailing=True)))
assert TS.print(unshare(S.Program(body=[function]))) == """function f() {
    const a = () => {
        return class {
            b() {
                c;
            }
        };
    }; // trailing
}
"""
assert TS.print(unshare(block(S.Comment(text=" only", trailing=True)))) == "{ // only\n}"
assert TS.print(unshare(block(S.Comment(block=True, text=" only ", trailing=True)))) == "{ /* only */\n}"
assert TS.print(unshare(S.TSEnumDeclaration(id=n("E"), body=S.TSEnumBody(members=[
    S.Comment(text=" a", trailing=True), S.TSEnumMember(id=a), S.Comment(text=" b"), S.Comment(text=" c",
                                                                                                trailing=True)])))) == (
    "enum E { // a\n    a,\n    // b // c\n}")
literal = S.TSTypeLiteral(members=[S.Comment(text=" a"), S.TSPropertySignature(key=a)])
statement = S.TSTypeAliasDeclaration(id=n("X"), typeAnnotation=literal)
namespace = S.TSModuleDeclaration(moduleKind="namespace", id=n("N"), body=S.TSModuleBlock(body=[statement]))
assert TS.print(unshare(namespace)) == (
    "namespace N {\n    type X = {\n        // a\n        a;\n    };\n}")
assert TS.print(unshare(S.Program(hashbang="/usr/bin/env node", body=[S.Comment(text=" a"), say("b")]))) == (
    "#!/usr/bin/env node\n// a\nb;\n")
assert TS.print(unshare(S.Program(body=[S.Comment(text=" first", trailing=True)]))) == "// first\n"
print("ok")

## TSPRT-06 · Printing validates the tree and checks it against the standard; JSX's arrow functions

In [ ]:
with raises(PrintError, match="a SequenceExpression needs two expressions"):
    TS.print(unshare(S.SequenceExpression(expressions=[a])))
with raises(PrintError, match="TSAnyKeyword is not ECMAScript, but this is ES2025"):
    ECMAScript2025.print(S.TSAnyKeyword())
with raises(PrintError, match="JSXText needs JSX, but this is TypeScript 5.9"):
    TS.print(unshare(S.JSXText(raw="a")))
with raises(PrintError, match="TSTypeAssertion is not allowed with JSX, but this is TypeScript 5.9 with JSX"):
    TSX.print(unshare(S.TSTypeAssertion(typeAnnotation=T, expression=a)))
generic = S.ArrowFunctionExpression(typeParameters=S.TSTypeParameterDeclaration(params=[
    S.TSTypeParameter(name=n("X"), default=S.TSTypeReference(
        typeName=n("A"), typeArguments=S.TSTypeParameterInstantiation(
        params=[T])))]), params=[a], body=a)
assert TSX.print(unshare(generic)) == "<X = A<T>,>(a) => a" and TS.print(unshare(generic)) == "<X = A<T>>(a) => a"
constrained = S.ArrowFunctionExpression(typeParameters=S.TSTypeParameterDeclaration(params=[
    S.TSTypeParameter(name=n("X"), constraint=T)]), body=a)
assert TSX.print(unshare(constrained)) == "<X extends T>() => a"
reparsed = TSX.parse(TSX.print(unshare(S.Program(body=[S.ExpressionStatement(expression=generic)]))))
assert reparsed.body[0].expression.params
print("ok")